# Task 2: Plant Disease Detection (PlantVillage)
**Goal:** Identify the plant type and disease from a photo of a leaf (multi-class image classification).

**Dataset:** PlantVillage dataset, Kaggle (leaf images of healthy and diseased crops such as tomato, potato, pepper and apple).

**Approach:** Transfer learning with **MobileNetV2** (pre-trained on ImageNet), data augmentation and a short fine-tuning phase. Evaluated with accuracy, macro F1, a classification report and a confusion matrix.

**Before running in Colab:** Runtime, Change runtime type, choose **T4 GPU**. Put your dataset zip in Google Drive.

## 1. Get the dataset into Colab
Upload the zip to Google Drive first (much faster than uploading a large file into Colab directly), then change `ZIP_PATH` to your file's name.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/archive.zip"   # <-- change to your zip file name
!cp "{ZIP_PATH}" /content/data.zip
!unzip -q -o /content/data.zip -d /content/data
print("Done")

## 2. Import libraries

In [ ]:
import os, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

SEED = 42
tf.keras.utils.set_random_seed(SEED)
IMG_SIZE = (160, 160)
BATCH = 32
MAX_PER_CLASS = 1000   # cap images per class so training finishes quickly; set to None to use everything
print("TensorFlow:", tf.__version__)
print("GPU found:", tf.config.list_physical_devices("GPU"))

## 3. Find the class folders
Each folder is one class (for example `Tomato___Late_blight`). Some versions of this dataset contain several copies (color, grayscale, segmented); the code picks the **color** images.

In [ ]:
DATA_DIR = "/content/data"
EXTS = (".jpg", ".jpeg", ".png")

def has_images(folder):
    return any(f.lower().endswith(EXTS) for f in os.listdir(folder))

def find_class_root(base):
    best, best_score = None, (-1, -1)
    for dirpath, dirnames, _ in os.walk(base):
        if "__MACOSX" in dirpath:
            continue
        subs = [d for d in dirnames if has_images(os.path.join(dirpath, d))]
        if len(subs) >= 5:
            low = dirpath.lower()
            preferred = 0 if ("grayscale" in low or "segmented" in low) else 1
            score = (preferred, len(subs))
            if score > best_score:
                best, best_score = dirpath, score
    if best is None:
        raise FileNotFoundError("Could not find class folders. Check the unzip step.")
    return best

root = find_class_root(DATA_DIR)
class_names = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)) and has_images(os.path.join(root, d)))
num_classes = len(class_names)
print("Dataset folder:", root)
print("Number of classes:", num_classes)
print(class_names)

## 4. Build the image table (and cap very large classes)

In [ ]:
rows = []
for idx, cls in enumerate(class_names):
    files = [f for f in glob.glob(os.path.join(root, cls, "*")) if f.lower().endswith(EXTS)]
    random.Random(SEED).shuffle(files)
    if MAX_PER_CLASS:
        files = files[:MAX_PER_CLASS]
    rows += [(f, idx) for f in files]

df = pd.DataFrame(rows, columns=["path", "label"])
print("Total images used:", len(df))

counts = df.label.map(dict(enumerate(class_names))).value_counts().sort_values()
counts.plot(kind="barh", figsize=(8, max(4, num_classes * 0.25)), title="Images per class")
plt.xlabel("Images"); plt.tight_layout(); plt.show()

## 5. Look at some leaves

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, row) in zip(axes.ravel(), df.sample(8, random_state=SEED).iterrows()):
    ax.imshow(plt.imread(row.path))
    ax.set_title(class_names[row.label].replace("___", "\n"), fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 6. Split into train (70%), validation (15%) and test (15%)
The split is **stratified**, so every class appears in all three parts.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df.label, random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

## 7. Build the input pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

def make_ds(d, training=False):
    ds = tf.data.Dataset.from_tensor_slices((d.path.values, d.label.values.astype("int32")))
    if training:
        ds = ds.shuffle(len(d), seed=SEED)
    return ds.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_df, training=True)
val_ds = make_ds(val_df)
test_ds = make_ds(test_df)

## 8. Build the model (MobileNetV2 transfer learning)

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,),
                                         include_top=False, weights="imagenet")
base.trainable = False                          # freeze the pre-trained layers first

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = layers.RandomFlip("horizontal_and_vertical")(inputs)   # augmentation (only active during training)
x = layers.RandomRotation(0.15)(x)
x = layers.RandomZoom(0.1)(x)
x = layers.Rescaling(1./127.5, offset=-1)(x)    # MobileNetV2 expects pixels in [-1, 1]
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)
model = models.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 9. Train: phase 1 (only the new top layers)

In [ ]:
es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=4, callbacks=[es])

## 10. Train: phase 2 (fine-tune the last layers of MobileNetV2 with a small learning rate)

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=3, callbacks=[es])

## 11. Training curves

In [ ]:
def join(key):
    return h1.history[key] + h2.history[key]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(join("accuracy"), label="train"); ax[0].plot(join("val_accuracy"), label="validation")
ax[0].axvline(len(h1.history["accuracy"]) - 1, color="gray", linestyle="--", label="fine-tuning starts")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(join("loss"), label="train"); ax[1].plot(join("val_loss"), label="validation")
ax[1].axvline(len(h1.history["loss"]) - 1, color="gray", linestyle="--")
ax[1].set_title("Loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.show()

## 12. Evaluate on the test set

In [ ]:
y_true = test_df.label.values
y_pred = np.argmax(model.predict(test_ds), axis=1)

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"Test accuracy: {acc:.4f} | Macro F1: {macro_f1:.4f}\n")
print(classification_report(y_true, y_pred, labels=list(range(num_classes)),
                            target_names=class_names, digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap="Blues", xticklabels=False, yticklabels=False)
plt.xlabel("Predicted class"); plt.ylabel("Actual class")
plt.title("Confusion matrix (test set)")
plt.show()

per_class = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
worst = np.argsort(per_class)[:5]
print("Hardest classes (lowest recall):")
for i in worst:
    print(f"  {class_names[i]}: {per_class[i]:.2%}")

## 13. Sample predictions

In [ ]:
sample = test_df.sample(8, random_state=1)
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    img = tf.image.resize(tf.io.decode_image(tf.io.read_file(row.path), channels=3, expand_animations=False), IMG_SIZE)
    p = model.predict(tf.expand_dims(img, 0), verbose=0)[0]
    pred = int(np.argmax(p))
    ax.imshow(img.numpy().astype("uint8"))
    ax.set_title(f"Actual: {class_names[row.label]}\nPred: {class_names[pred]} ({p[pred]:.2f})".replace("___", " - "),
                 fontsize=8, color="green" if pred == row.label else "red")
    ax.axis("off")
plt.tight_layout(); plt.show()

## 14. Results and Conclusion
*(Fill in the blanks from the outputs of section 12 after running the notebook.)*

| Metric | Value |
|--------|-------|
| Number of classes | ____ |
| Test accuracy | ____ |
| Macro F1 | ____ |

**Observations:**
- Transfer learning with MobileNetV2 reached ____% accuracy on the unseen test set across ____ classes.
- The hardest classes were ____ (see the "Hardest classes" list), probably because their symptoms look similar to other diseases.
- Training and validation curves ____ (close together / diverging), which suggests the model is ____ (generalizing well / overfitting).

**Conclusion:** A pre-trained MobileNetV2 with augmentation and fine-tuning can recognize plant diseases from leaf photos with high accuracy. PlantVillage photos were taken against plain backgrounds in a lab setting, so accuracy on real farm photos would likely be lower.

**Future work:** Train on all images instead of the per-class cap, test on real-field photos, try EfficientNet, and deploy the model as a simple web app.